<a href="https://colab.research.google.com/github/Div4567/ResearchProject/blob/main/bert-hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies
!pip install torch torch-geometric transformers scikit-learn pandas numpy scipy tqdm

# Download data
!wget https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip ml-1m.zip

# Download your script
!wget https://raw.githubusercontent.com/YOUR_REPO/movie_recommender.py

# Run training
!python movie_recommender.py --use-cuda --num-epochs 50

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00
--2026-06-13 06:43:38--  https://files.grouplens.org/datasets/movielens/ml-1m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5917549 (5.6M) [application/zip]
Saving to: ‘ml-1m.zip’

ml-1m.zip           100%[===================>]   5.64M  15.7MB/s    in 0.4s    

2026-06-13 06:43:39 (15.7 MB/s) - ‘ml-1m.zip’ saved [5917549/5917549]

Archive:  ml-1m.zip
   creating: ml-1m/
  inflating: ml-1m/movies.dat        
  inflating: ml-1m/ratings.dat       
  inflating: ml-1m/README            
  inflating: ml-1m/users.dat         
--2026-06-13 06:43:39--  https://raw.githubusercontent.com/YOUR_REPO/movie_recommender.py
Resolving raw.githubusercontent.com (raw.githubusercont

In [2]:
"""
Hybrid Movie Recommendation System: LightGCN + BERT
Combines graph neural networks with semantic text understanding
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Handle torch_geometric version compatibility
try:
    from torch_geometric.utils import from_scipy_sparse_tensor
except ImportError:
    # For newer versions of torch_geometric
    def from_scipy_sparse_tensor(sparse_tensor):
        coo = sparse_tensor.tocoo()
        indices = torch.from_numpy(np.array([coo.row, coo.col])).long()
        return indices, None
import pandas as pd
from scipy.sparse import csr_matrix
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
import argparse
from tqdm import tqdm


# ============================================================================
# PART 1: DATA LOADING & PREPROCESSING
# ============================================================================

class MovieLensLoader:
    """Load and preprocess MovieLens dataset"""

    def __init__(self, data_path='.'):
        self.data_path = Path(data_path)

    def load_data(self):
        """Load ratings and movies data"""
        # Load ratings
        self.ratings = pd.read_csv(
            self.data_path / 'ratings.dat',
            sep='::',
            header=None,
            names=['userId', 'movieId', 'rating', 'timestamp'],
            engine='python'
        )

        # Load movies
        self.movies = pd.read_csv(
            self.data_path / 'movies.dat',
            sep='::',
            header=None,
            names=['movieId', 'title', 'genres'],
            engine='python',
            encoding='latin-1'
        )

        print(f"Loaded {len(self.ratings)} ratings for {self.ratings['movieId'].nunique()} movies")
        return self.ratings, self.movies

    def create_mappings(self):
        """Create user and movie ID mappings"""
        self.user_map = {uid: idx for idx, uid in enumerate(sorted(self.ratings['userId'].unique()))}
        self.movie_map = {mid: idx for idx, mid in enumerate(sorted(self.movies['movieId'].unique()))}
        self.reverse_movie_map = {v: k for k, v in self.movie_map.items()}

        return self.user_map, self.movie_map

    def build_interaction_matrix(self):
        """Build sparse user-item interaction matrix"""
        self.ratings['user_idx'] = self.ratings['userId'].map(self.user_map)
        self.ratings['movie_idx'] = self.ratings['movieId'].map(self.movie_map)

        n_users = len(self.user_map)
        n_movies = len(self.movie_map)

        self.interaction_matrix = csr_matrix(
            (np.ones(len(self.ratings)),
             (self.ratings['user_idx'].values, self.ratings['movie_idx'].values)),
            shape=(n_users, n_movies)
        )

        sparsity = 1 - self.interaction_matrix.nnz / (n_users * n_movies)
        print(f"Interaction matrix: {n_users} x {n_movies}, Sparsity: {sparsity:.4f}")

        return self.interaction_matrix

    def prepare_text_features(self, tokenizer_name='bert-base-uncased'):
        """Tokenize movie titles and genres for BERT"""
        self.movies['text'] = self.movies['title'] + " [" + self.movies['genres'] + "]"

        tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        movie_texts = self.movies.sort_values('movieId')['text'].tolist()

        tokens = tokenizer(
            movie_texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )

        print(f"Tokenized {len(movie_texts)} movie descriptions")
        return tokens, tokenizer

    def train_val_test_split(self, train_ratio=0.6, val_ratio=0.2):
        """Split data into train/val/test sets"""
        shuffled = self.ratings.sample(frac=1, random_state=42)
        n = len(shuffled)

        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        self.train_data = shuffled.iloc[:n_train]
        self.val_data = shuffled.iloc[n_train:n_train+n_val]
        self.test_data = shuffled.iloc[n_train+n_val:]

        print(f"Train: {len(self.train_data)}, Val: {len(self.val_data)}, Test: {len(self.test_data)}")


# ============================================================================
# PART 2: MODELS
# ============================================================================

class LightGCN(nn.Module):
    """Light Graph Convolution Network for collaborative filtering"""

    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # Initialize embeddings
        self.user_embedding = nn.Parameter(
            torch.nn.init.xavier_uniform_(torch.empty(num_users, embedding_dim))
        )
        self.item_embedding = nn.Parameter(
            torch.nn.init.xavier_uniform_(torch.empty(num_items, embedding_dim))
        )

    def forward(self, edge_index):
        """Forward pass with multi-layer propagation"""
        # Concatenate user and item embeddings
        embeddings = [torch.cat([self.user_embedding, self.item_embedding], dim=0)]

        x = embeddings[0]
        for _ in range(self.num_layers):
            x = self._propagate(edge_index, x)
            embeddings.append(x)

        # Average across layers
        final_emb = sum(embeddings) / len(embeddings)

        user_emb = final_emb[:self.num_users]
        item_emb = final_emb[self.num_users:]

        return user_emb, item_emb

    def _propagate(self, edge_index, x):
        """Message passing aggregation"""
        row, col = edge_index

        # Compute normalization: 1/sqrt(d_i * d_j)
        num_nodes = x.size(0)
        deg = torch.zeros(num_nodes, device=x.device)
        deg.scatter_add_(0, col, torch.ones(edge_index.size(1), device=x.device))

        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Aggregate messages
        out = torch.zeros_like(x)
        out.scatter_add_(0, col.unsqueeze(1).expand(-1, x.size(1)),
                        norm.view(-1, 1) * x[row])

        return out


class BERTMovieEncoder(nn.Module):
    """BERT-based text encoder for movies"""

    def __init__(self, hidden_dim=64, pretrained='bert-base-uncased'):
        super().__init__()
        self.bert = AutoModel.from_pretrained(pretrained)
        self.projection = nn.Linear(768, hidden_dim)

        # FIX 1: Add the cache here instead
        self.cached_cls = None

        # Freeze BERT weights for efficiency
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        """Encode movie text to embeddings"""

        # FIX 2: Only run the heavy BERT model once
        if self.cached_cls is None:
            with torch.no_grad():
                outputs = self.bert(input_ids, attention_mask=attention_mask)
                # Extract [CLS] token and .detach() it to strictly sever it from any old graphs
                self.cached_cls = outputs.last_hidden_state[:, 0, :].detach()

        # FIX 3: Always run the projection layer so it gets a fresh graph every batch!
        embeddings = self.projection(self.cached_cls)

        return embeddings
class HybridRecommender(nn.Module):
    """Hybrid recommender combining LightGCN and BERT"""

    def __init__(self, num_users, num_items, embedding_dim=64,
                 alpha=0.5, bert_model='bert-base-uncased'):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.alpha = alpha

        # Components
        self.lightgcn = LightGCN(num_users, num_items, embedding_dim)
        self.bert_encoder = BERTMovieEncoder(embedding_dim, bert_model)

        # Fusion layer for combining embeddings
        self.fusion = nn.Sequential(
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embedding_dim, embedding_dim)
        )

    def forward(self, edge_index, movie_input_ids, movie_attention_mask):
        """Forward pass returning user and item embeddings"""
        # Collaborative embeddings
        user_emb_gcn, item_emb_gcn = self.lightgcn(edge_index)

        # Content embeddings (the caching is handled safely inside here now!)
        item_emb_bert = self.bert_encoder(movie_input_ids, movie_attention_mask)

        # Hybrid combination
        item_emb_hybrid = (
            self.alpha * item_emb_gcn +
            (1 - self.alpha) * item_emb_bert
        )

        return user_emb_gcn, item_emb_hybrid, item_emb_gcn, item_emb_bert
# ============================================================================
# PART 3: TRAINING
# ============================================================================

class RatingDataset(Dataset):
    """Rating dataset with negative sampling"""

    def __init__(self, ratings_df, num_items, neg_samples=1):
        self.ratings = ratings_df.reset_index(drop=True)
        self.num_items = num_items
        self.neg_samples = neg_samples

        # Get positive items per user
        self.user_items = {}
        for user_idx, movie_idx in zip(self.ratings['user_idx'], self.ratings['movie_idx']):
            if user_idx not in self.user_items:
                self.user_items[user_idx] = set()
            self.user_items[user_idx].add(movie_idx)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        row = self.ratings.iloc[idx]
        user_id = int(row['user_idx'])
        pos_item = int(row['movie_idx'])

        # Negative sampling - sample items not interacted with
        neg_items = []
        for _ in range(self.neg_samples):
            while True:
                neg_item = np.random.randint(0, self.num_items)
                if neg_item not in self.user_items.get(user_id, set()):
                    neg_items.append(neg_item)
                    break

        return {
            'user_id': torch.tensor(user_id, dtype=torch.long),
            'pos_item': torch.tensor(pos_item, dtype=torch.long),
            'neg_item': torch.tensor(neg_items[0], dtype=torch.long)
        }


class BPRLoss(nn.Module):
    """Bayesian Personalized Ranking loss"""

    def forward(self, user_emb, pos_item_emb, neg_item_emb):
        """BPR loss: maximize margin between positive and negative"""
        pos_scores = torch.sum(user_emb * pos_item_emb, dim=1)
        neg_scores = torch.sum(user_emb * neg_item_emb, dim=1)

        loss = -torch.mean(torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-5))
        return loss


def train_epoch(model, train_loader, optimizer, loss_fn, edge_index,
                movie_input_ids, movie_attention_mask, device):
    """Train one epoch"""
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc='Training'):
        user_ids = batch['user_id'].to(device)
        pos_items = batch['pos_item'].to(device)
        neg_items = batch['neg_item'].to(device)

        optimizer.zero_grad()

        # Forward pass
        user_emb, item_emb_hybrid, _, _ = model(
            edge_index,
            movie_input_ids,
            movie_attention_mask
        )

        # Get batch embeddings
        user_batch = user_emb[user_ids]
        pos_batch = item_emb_hybrid[pos_items]
        neg_batch = item_emb_hybrid[neg_items]

        # Compute loss
        loss = loss_fn(user_batch, pos_batch, neg_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(model, val_ratings, edge_index, movie_input_ids,
             movie_attention_mask, device, k=10):
    """Evaluate on validation set"""
    model.eval()

    user_emb, item_emb, _, _ = model(edge_index, movie_input_ids, movie_attention_mask)

    recall_scores = []
    for user_id in val_ratings['user_idx'].unique():
        user_items = val_ratings[val_ratings['user_idx'] == user_id]['movie_idx'].values

        # Get top-k recommendations
        user_vec = user_emb[user_id].unsqueeze(0)
        scores = torch.matmul(user_vec, item_emb.t()).squeeze()

        _, top_indices = torch.topk(scores, k)
        top_items = top_indices.cpu().numpy()

        # Compute recall
        hits = len(set(top_items) & set(user_items))
        recall = hits / len(user_items) if len(user_items) > 0 else 0
        recall_scores.append(recall)

    return np.mean(recall_scores)


# ============================================================================
# PART 4: MAIN TRAINING SCRIPT
# ============================================================================

def main(args):
    device = torch.device('cuda' if torch.cuda.is_available() and args.use_cuda else 'cpu')
    print(f"Using device: {device}")

    # Load data
    print("\n=== Loading Data ===")
    loader = MovieLensLoader(args.data_path)
    ratings, movies = loader.load_data()
    loader.create_mappings()
    loader.build_interaction_matrix()
    tokens, tokenizer = loader.prepare_text_features()
    loader.train_val_test_split()

    # Convert edge index for GCN
    n_users = len(loader.user_map)
    n_items = len(loader.movie_map)

    edge_index, _ = from_scipy_sparse_tensor(loader.interaction_matrix)
    edge_index = edge_index.to(device)

    # Move tokens to device
    movie_input_ids = tokens['input_ids'].to(device)
    movie_attention_mask = tokens['attention_mask'].to(device)

    # Create model
    print("\n=== Building Model ===")
    model = HybridRecommender(
        num_users=n_users,
        num_items=n_items,
        embedding_dim=args.embedding_dim,
        alpha=args.alpha
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
    loss_fn = BPRLoss()

    # Create data loader
    train_dataset = RatingDataset(loader.train_data, n_items)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)

    # Training loop
    print("\n=== Training ===")
    best_recall = 0
    for epoch in range(args.num_epochs):
        train_loss = train_epoch(
            model, train_loader, optimizer, loss_fn,
            edge_index, movie_input_ids, movie_attention_mask,
            device
        )

        # Validate
        if (epoch + 1) % args.eval_every == 0:
            val_recall = evaluate(
                model, loader.val_data, edge_index,
                movie_input_ids, movie_attention_mask, device,
                k=args.k
            )

            print(f"Epoch {epoch+1}/{args.num_epochs} | "
                  f"Loss: {train_loss:.4f} | "
                  f"Recall@{args.k}: {val_recall:.4f}")

            if val_recall > best_recall:
                best_recall = val_recall
                torch.save(model.state_dict(), 'best_model.pt')

    # Test evaluation
    print("\n=== Testing ===")
    model.load_state_dict(torch.load('best_model.pt'))
    test_recall = evaluate(
        model, loader.test_data, edge_index,
        movie_input_ids, movie_attention_mask, device,
        k=args.k
    )
    print(f"Test Recall@{args.k}: {test_recall:.4f}")


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-path', default='ml-1m', help='Path to MovieLens data')
    parser.add_argument('--embedding-dim', type=int, default=64)
    parser.add_argument('--alpha', type=float, default=0.5, help='Weight for GCN vs BERT')
    parser.add_argument('--batch-size', type=int, default=256)
    parser.add_argument('--learning-rate', type=float, default=0.001)
    parser.add_argument('--num-epochs', type=int, default=50)
    parser.add_argument('--eval-every', type=int, default=5)
    parser.add_argument('--k', type=int, default=10, help='Recall@K')
    parser.add_argument('--use-cuda', action='store_true')

    # FIX: Use parse_known_args() instead of parse_args()
    # This captures your arguments and safely ignores the Jupyter '-f' argument
    args, unknown = parser.parse_known_args(args=['--data-path', '/','--use-cuda','--num-epochs', '10',    # <-- Set total epochs to 10
        '--eval-every', '5'])

    main(args)

Using device: cuda

=== Loading Data ===
Loaded 1000209 ratings for 3706 movies
Interaction matrix: 6040 x 3883, Sparsity: 0.9574


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...


=== Training ===


Training: 100%|██████████| 2345/2345 [02:18<00:00, 16.91it/s]


Epoch 5/10 | Loss: 0.2964 | Recall@10: 0.0515


Training: 100%|██████████| 2345/2345 [02:17<00:00, 17.01it/s]


Epoch 10/10 | Loss: 0.2530 | Recall@10: 0.0536

=== Testing ===
Test Recall@10: 0.0513


In [ ]:
from google.colab import drive
drive.mount('/content/drive')